<a href="https://colab.research.google.com/github/jeandecamps-prog/AI-for-chemistry-ML/blob/main/LLM_GPT_2_ET_Charges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import numpy as np
import pandas as pd

# 1. Télécharger l'archive brute (1.1 Go)
# C'est l'adresse directe du dépôt ESNUEL
!wget -O data.tar.xz https://sid.erda.dk/share_redirect/c7LF5NaYvH

# 2. Extraire l'archive
# Cela va créer un dossier nommé 'data' dans ton environnement Colab
!tar -xf data.tar.xz

# 3. Charger le fichier principal des électrophiles (format compressé interne)
# On pointe vers le dossier extrait 'data/QMdata4ML/'
path_elec = 'data/QMdata4ML/df_elec.csv.gz'

# Chemin vers le fichier des electrophiles

path_elec = 'data/QMdata4ML/df_elec.csv.gz'

# Pandas gère la décompression (.gz) automatiquement
df_elec = pd.read_csv(path_elec)

--2026-04-30 21:24:01--  https://sid.erda.dk/share_redirect/c7LF5NaYvH
Resolving sid.erda.dk (sid.erda.dk)... 130.225.104.13
Connecting to sid.erda.dk (sid.erda.dk)|130.225.104.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1170060920 (1.1G)
Saving to: ‘data.tar.xz’

data.tar.xz         100%[===================>]   1.09G  16.7MB/s    in 77s     

2026-04-30 21:25:19 (14.6 MB/s) - ‘data.tar.xz’ saved [1170060920/1170060920]

tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.FinderInfo'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemTextContentLanguage'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseVersion'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.metadata:kMDItemKeyphraseLabels'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.

In [ ]:
df=df_elec
print(df.head(10))

   Unnamed: 0                              smiles  elec_sites   elec_names  \
0           0                      NOCc1cccc(I)c1           3  double_bond   
1           1                      NOCc1cccc(I)c1           4  double_bond   
2           2                      NOCc1cccc(I)c1           5  double_bond   
3           3                      NOCc1cccc(I)c1           6  double_bond   
4           4                      NOCc1cccc(I)c1           7  double_bond   
5           5                      NOCc1cccc(I)c1           9  double_bond   
6           6  CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl           8  double_bond   
7           7  CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl          12  double_bond   
8           8  CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl          13  double_bond   
9           9  CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl          14  double_bond   

   MAA_values                                     elec_GCS_3_cm5          Set  
0   90.348433  [-0.01706, 0.12057, -0.11146, -0.08969, 0.0, -

In [ ]:
import ast
import pandas as pd

# 1. On redéfinit la fonction vitale pour lire les charges quantiques
def parse_gcs(x):
    try: return ast.literal_eval(x) if isinstance(x, str) else x
    except: return []

# On s'assure que les charges quantiques sont bien sous forme de vraie liste Python
df['elec_GCS_3_cm5'] = df['elec_GCS_3_cm5'].apply(parse_gcs)

# 2. Fonction pour créer la phrase (le Prompt)
def build_prompt(row):
    # On arrondit un peu les charges pour ne pas noyer le modèle sous les décimales
    charges_arrondies = [round(c, 4) for c in row['elec_GCS_3_cm5']]

    instruction = (
        f"SMILES: {row['smiles']} | "
        f"Site: {row['elec_sites']} | "
        f"Charges GCS_3: {charges_arrondies} | "
        f"MAA: "
    )

    reponse = str(round(row['MAA_values'], 2))
    texte_complet = instruction + reponse

    return instruction, texte_complet

# 3. On applique la fonction à tout le dataset 'df'
df[['prompt_question', 'prompt_complet']] = df.apply(
    lambda row: pd.Series(build_prompt(row)), axis=1
)

print("✅ Préparation terminée ! Voici le premier prompt :")
print(df['prompt_complet'].iloc[0])

✅ Préparation terminée ! Voici le premier prompt :
SMILES: NOCc1cccc(I)c1 | Site: 3 | Charges GCS_3: [-0.0171, 0.1206, -0.1115, -0.0897, 0.0, -0.4566, 0.1066, 0.0894, 0.0149, 0.0858, 0.0, -0.0848, 0.0994, 0.0, 0.0, 0.0, 0.0, -0.677, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0048, -0.105, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0912, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MAA: 90.35


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

print("1️⃣ Chargement des données...")
# Assure-toi que ton DataFrame avec la colonne 'prompt_question' et 'MAA_values' est prêt ici.
# On va utiliser le df_final qu'on a construit tout à l'heure.
# Pour l'exemple, supposons que tu l'as sauvegardé en CSV et réimporté :
# df_final = pd.read_csv('mon_dataset_prompts.csv')

# On sépare en Train/Test (comme pour XGBoost)
train_df = df[df['Set'].str.startswith('Train')]
test_df = df[df['Set'].str.startswith('Test')]

# On convertit pour Hugging Face
train_dataset = Dataset.from_pandas(train_df[['prompt_question', 'MAA_values']])
test_dataset = Dataset.from_pandas(test_df[['prompt_question', 'MAA_values']])

print("2️⃣ Préparation du Tokenizer et du Modèle (GPT-2)...")
model_name = "gpt2" # On peut changer pour un autre modèle plus tard
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # GPT-2 a besoin d'un token de padding

# On charge le modèle avec 1 seul label en sortie (pour la régression)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)
model.config.pad_token_id = tokenizer.pad_token_id

print("3️⃣ Configuration de LoRA (La technique d'accélération)...")
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Classification de séquence (ici régression)
    r=8, # La taille du "greffon"
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"] # Les couches de GPT-2 qu'on va modifier
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() # Tu vas voir, c'est minime !

print("4️⃣ Tokenization des textes...")
def tokenize_function(examples):
    # On transforme le texte en nombres pour le modèle
    return tokenizer(examples["prompt_question"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# On renomme 'MAA_values' en 'labels' car le Trainer exige ce nom
tokenized_train = tokenized_train.rename_column("MAA_values", "labels")
tokenized_test = tokenized_test.rename_column("MAA_values", "labels")


print("5️⃣ Définition des métriques de succès...")
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Calculs classiques
    r2 = r2_score(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))

    # Calcul du RMSE relatif (en pourcentage)
    moyenne_maa = np.mean(labels)
    rrmse_pourcentage = (rmse / moyenne_maa) * 100

    return {
        "r2": r2,
        "rmse": rmse,
        "rrmse_%": rrmse_pourcentage
    }

print("6️⃣ Lancement de l'Entraînement ! 🚀")
training_args = TrainingArguments(
    output_dir="./resultats_gpt2_maa",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # Commence petit pour tester
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True, # Utilise la précision mixte pour aller encore plus vite sur GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

print("✅ Entraînement terminé ! Voici l'évaluation finale :")
print(trainer.evaluate())

1️⃣ Chargement des données...
2️⃣ Préparation du Tokenizer et du Modèle (GPT-2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


3️⃣ Configuration de LoRA (La technique d'accélération)...
trainable params: 295,680 || all params: 124,736,256 || trainable%: 0.2370
4️⃣ Tokenization des textes...


Map:   0%|          | 0/454001 [00:00<?, ? examples/s]

Map:   0%|          | 0/80118 [00:00<?, ? examples/s]

5️⃣ Définition des métriques de succès...
6️⃣ Lancement de l'Entraînement ! 🚀


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# On s'assure que la fonction de métrique est bien en mémoire (Étape 5)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    r2 = r2_score(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    return {"r2": r2, "rmse": rmse}

print("6️⃣ Lancement de l'Entraînement ! 🚀 (Reprise)")

# CORRECTION : eval_strategy au lieu de evaluation_strategy
training_args = TrainingArguments(
    output_dir="./resultats_gpt2_maa",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch", # <--- Le correctif est ici !
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
)

trainer = Trainer(
    model=model, # Ton modèle de l'étape 3 est toujours là
    args=training_args,
    train_dataset=tokenized_train, # Tes données de l'étape 4 sont toujours là !
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# On allume le moteur !
trainer.train()

print("✅ Entraînement terminé ! Voici l'évaluation finale :")
print(trainer.evaluate())

6️⃣ Lancement de l'Entraînement ! 🚀 (Reprise)


Epoch,Training Loss,Validation Loss,R2,Rmse
1,1421.504980,1340.134766,0.740253,36.607851
2,913.017578,1003.325684,0.805534,31.675317
3,1091.537891,951.408875,0.815597,30.844917


✅ Entraînement terminé ! Voici l'évaluation finale :


{'eval_loss': 951.4088745117188, 'eval_r2': 0.8155965209007263, 'eval_rmse': 30.84491651004617, 'eval_runtime': 274.9586, 'eval_samples_per_second': 291.382, 'eval_steps_per_second': 18.214, 'epoch': 3.0}


In [1]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Calculs classiques
    r2 = r2_score(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))

    # Calcul du RMSE relatif (en pourcentage)
    moyenne_maa = np.mean(labels)
    rrmse_pourcentage = (rmse / moyenne_maa) * 100

    return {
        "r2": r2,
        "rmse": rmse,
        "rrmse_%": rrmse_pourcentage
    }